In [1]:
import torch
import re
import gc 

import pandas as pd
import numpy as np

from collections import defaultdict
from tqdm.notebook import tqdm
tqdm.pandas()

from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
triples = pd.read_excel("../outputs/clean_outputs/triples_coalesced.xlsx").drop("Unnamed: 0", axis=1)
triples.head()

,id,company,job title,text,triples_qwen_structured,triples_qwen_semi-structured,triples_qwen_unstructured,triples_gemma_structured,triples_gemma_semi-structured,triples_gemma_unstructured,triples_llama_structured,triples_llama_semi-structured,triples_llama_unstructured
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('IT-administrator', 'REQUIRES_SKILL', 'setup...","[('IT-administrator', 'REQUIRES_SKILL', 'netwo...","[('IT-administrator', 'requires', 'technical_s...","[('IT-administrator', 'INVOLVES_TASK', 'Mainta...","[('IT-administrator', 'REQUIRES_SKILL', 'Syste...",[('IT-administrator – få indflydelse på et set...,"[('IT-administrator', 'REQUIRES_SKILL', 'setup...","[('IT-administrator', 'REQUIRES_SKILL', 'Docke...","[('IT-administrator', 'har', 'få indflydelse p..."
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('IT-administrator', 'REQUIRES_SKILL', 'setup...","[('IT-administrator', 'REQUIRES_SKILL', 'netwo...","[('IT-administrator', 'requires', 'technical_s...","[('IT-administrator', 'REQUIRES_SKILL', 'IT Ad...","[('IT-administrator', 'REQUIRES_SKILL', 'Syste...",[('IT-administrator – få indflydelse på et set...,"[('IT-administrator', 'REQUIRES_SKILL', 'Docke...",[('IT-administrator – få indflydelse på et set...,"[('IT-administrator', 'har', 'få indflydelse p..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...,"[('SQE Manager', 'REQUIRES_SKILL', 'Root Cause...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality Ma...","[('SQE Manager', 'requires', 'quality assuranc...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'REQUIRES_SKILL', 'Quality As...","[('SQE Manager', 'title', 'SQE Manager'), ('SQ...","[('SQE Manager', 'REQUIRES_QUALITY', 'detail-o...","[('SQE Manager', 'REQUIRES_SKILL', 'DevOps Eng...","[('SQE Manager', 'requires', 'English language..."
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...,"[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'requires'...","[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'REQUIRES_...","[('Kundeservice / teknisk support', 'title', '...","[('Kundeservice', 'REQUIRES_QUALITY', 'detail-...","[('Kundeservice', 'REQUIRES_SKILL', 'Teknisk s...","[('Kundeservice', 'har', 'en teknisk support s..."
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...,"[('Retail Designer', 'REQUIRES_SKILL', 'Techni...","[('Retail Designer', 'REQUIRES_SKILL', 'Design...","[('Retail Designer', 'requires', 'technical sk...","[('Retail designer', 'REQUIRES_SKILL', 'teknik...","[('Retail designer', 'REQUIRES_SKILL', 'teknik...","[('Retail designer med teknikken på plads', 'h...","[('Retail designer med teknikken på plads', 'R...","[('Dansk designer', 'REQUIRES_QUALITY', 'SKILL...",[('Retail designer requires technisk kompetenc...


### Prepare ISCO data

In [3]:
def prepare_iscos(row):
    row["Included occupations"] = row["Included occupations"].replace('Examples of the occupations classified here:', '')
    row["Included occupations"] = "".join(row["Included occupations"].split("\n")[:4]).replace("- ", ", ")

    return {row['ISCO 08 Code']: f"{row['Title EN']}. {row['Definition']} (e.g.{row['Included occupations']})"}

In [4]:
iscos = pd.read_excel("ISCO-08.xlsx")
iscos = iscos[(iscos["ISCO 08 Code"].astype(str).str.fullmatch(r'\d{4}'))][["ISCO 08 Code", "Title EN", 
                                                                            "Included occupations", "Definition"]]
# TODO: Ugly, clean
result = iscos.apply(lambda row: prepare_iscos(row), axis=1).values

r_isco = {}

for d in result:
    r_isco.update(d)

### Embed ISCOs and triples

In [5]:
emb_model = SentenceTransformer('dajobbert-kg-specialized')

isco_embs = {}

# Compute embedding for both lists
for isco, text in r_isco.items():
    isco_embs[isco] = emb_model.encode(text, convert_to_tensor=True)    

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
isco_ids = list(isco_embs.keys())
isco_matrix = torch.stack([isco_embs[_id] for _id in isco_ids]).to("cuda:0")  # (N, d)

# Normalize once for cosine similarity
isco_matrix_norm = isco_matrix / isco_matrix.norm(dim=1, keepdim=True)

In [7]:
def find_matches(query_embedding, ids, matrix_norm, k=20):
    """
    query_embedding: torch.Tensor on cuda, shape [d]
    isco_ids: list of IDs (pre-built)
    isco_matrix_norm: pre-normalized matrix (N, d)
    k: number of matches to return
    """
    # Normalize query
    q = query_embedding / query_embedding.norm(dim=0, keepdim=True)

    # Compute cosine similarity (N)
    sims = torch.matmul(matrix_norm, q)

    # Top-k
    topk_vals, topk_idx = torch.topk(sims, k)

    return [ids[i] for i in topk_idx.tolist()]


In [8]:
for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        triples[f"triples_{model}_{prompt}_embedding"] = triples[f"triples_{model}_{prompt}"].progress_apply(
            lambda x: emb_model.encode(x, convert_to_tensor=True).to("cuda:0")
        )
        
        triples[f"triples_{model}_{prompt}_top_matches_isco"] = triples[f"triples_{model}_{prompt}_embedding"].progress_apply(
            lambda emb: find_matches(emb, isco_ids, isco_matrix_norm, k=15)
        )

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

...

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

### Embed ESCOs

In [10]:
df_skills = pd.read_csv("skills_en.csv")
df_skills["id"] = df_skills.index
df_skills = df_skills[["id", "skillType", "preferredLabel", "description"]]

df_skills.head()

,id,skillType,preferredLabel,description
0,0,skill/competence,manage musical staff,Assign and manage staff tasks in areas such as...
1,1,skill/competence,supervise correctional procedures,Supervise the operations of a correctional fac...
2,2,skill/competence,apply anti-oppressive practices,"Identify oppression in societies, economies, c..."
3,3,skill/competence,control compliance of railway vehicles regulat...,"Inspect rolling stock, components and systems ..."
4,4,skill/competence,identify available services,Identify the different services available for ...


In [11]:
df_skills.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13939 entries, 0 to 13938
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              13939 non-null  int64 
 1   skillType       13934 non-null  object
 2   preferredLabel  13939 non-null  object
 3   description     13939 non-null  object
dtypes: int64(1), object(3)
memory usage: 435.7+ KB


In [12]:
def prepare_escos(row):
    return {f"{row['id']:06d}" : f"{row['preferredLabel']}: {row['description']}"}

# TODO: Ugly, clean
result = df_skills.apply(lambda row: prepare_escos(row), axis=1).values

r_esco = {}

for d in result:
    r_esco.update(d)

In [18]:
embs_esco = {}

# Compute embedding for both lists
for esco, text in tqdm(r_esco.items()):
    embs_esco[esco] = emb_model.encode(text, convert_to_tensor=True)   

  0%|          | 0/13939 [00:00<?, ?it/s]

In [19]:
esco_ids = list(embs_esco.keys())
esco_matrix = torch.stack([embs_esco[_id] for _id in esco_ids]).to("cuda:0")  # (N, d)

# Normalize once for cosine similarity
esco_matrix_norm = esco_matrix / esco_matrix.norm(dim=1, keepdim=True)

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        triples[f"triples_{model}_{prompt}_top_matches_esco"] = triples[f"triples_{model}_{prompt}_embedding"].progress_apply(
            lambda emb: find_matches(emb, esco_ids, esco_matrix_norm, k=15)
        )

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

In [20]:
triples["triples_qwen_structured_top_matches_esco"].iloc[0]

['005386',
 '010026',
 '010923',
 '005810',
 '008163',
 '011116',
 '011151',
 '007348',
 '002990',
 '006552',
 '002764',
 '007239',
 '004901',
 '003053',
 '006337',
 '011009',
 '013736',
 '012640',
 '009103',
 '004542']

### Link ISCO/ESCO to triples

In [59]:
def run_pipeline(model, tokenizer, r_isco, r_esco, row, model_name, prompt_type):
    
    text = row[f"triples_{model_name}_{prompt_type}"]

    matches = {_id: r_isco[_id] for _id in row[f"triples_{model_name}_{prompt_type}_top_matches_isco"]}
    prompt1 = f"""
    Link the subjects and objects in these triples to their respective ISCO-08 codes (where applicable).
    A link takes the shape: (*subject/object*, has_isco, ISCO_XXXX) with XXXX replaced by the appropriate 4-digit code.
    Ensure that it matches this shape exactly. Only ever use has_isco as the predicate.

    You are given 20 possible ISCO codes and their definitions. When returning a triple, ensure you only use the code,
    not the definition. Precisely, you need to evaluate the triples you are provided, determine which ones match ISCO
    codes, and then return new triples containing the subject/object of the triples, has_isco, and then the ISCO code
    that matches it. If none of the triples match an isco definition, it is okay to return an empty list. Therefore,
    choose quality over quantity when it comes to matches. No match is better than a farfetched one. 
    
    Pick from the following ISCO codes/definitions:
    {matches}
    
    return the found links as a list of triples. Do not add any text to your output other than the subjects, 
    predicates, and objects. The output should therefore precisely be in the shape of [("s1", "p1", "o1"), ("s2", "p2", "o2"), ...]
    with the values replaced. 
    
    Here are the triples: 
    """

    matches2 = {_id: r_esco[_id] for _id in row[f"triples_{model_name}_{prompt_type}_top_matches_esco"]}
        
    prompt2 = f"""
    Link the subjects and objects in these triples to their respective ESCO skill codes (where applicable).
    A link takes the shape: (*skill/knowledge*, has_esco, ESCO_*code*) with *skill/knowledge* being replace by the 
    appropriate node and *code* replaced by the appropriate code. Only ever use has_esco as the predicate.
    Ensure that it matches this shape exactly. 

    You are given 20 possible ESCO codes and their definitions. When returning a triple, ensure you only use the code,
    not the definition. Precisely, you need to evaluate the triples you are provided, determine which ones match ESCO
    codes, and then return new triples containing the subject/object of the triples, has_isco, and then the ESCO code
    that matches it. If none of the triples match an esco definition, it is okay to return an empty list. Therefore,
    choose quality over quantity when it comes to matches. No match is better than a farfetched one. 
    
    Pick from the following ESCO codes/definitions:
    {matches2}
    
    return the found links as a list of triples. Do not add any text to your output other than the subjects, 
    predicates, and objects. The output should therefore precisely be in the shape of [("s1", "p1", "o1"), ("s2", "p2", "o2"), ...]
    with the values replaced. 

    Here are the triples:
    """

    if not text:
        return []

    output = {}
    
    for t, p in [("isco", prompt1), ("esco", prompt2)]:
        # Create message
        messages = [
            {"role": "user", "content": p + text}
        ]
    
        # Apply template
        processed_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    
        # Tokenize
        model_inputs = tokenizer([processed_text], return_tensors="pt").to(model.device)
    
        # conduct text completion
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096 # 16384
        )
    
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        output[t] = tokenizer.decode(output_ids, skip_special_tokens=True)
    
    # Return response
    return output

In [60]:
# del emb_model
del model

In [61]:
models = {"qwen" : "Qwen/Qwen3-4B-Instruct-2507",
          "llama" : "meta-llama/Llama-3.2-3B-Instruct",
          "gemma" : "google/gemma-3n-e4b-it"}

device = ("cuda:0" if torch.cuda.is_available() else "cpu")

final_results = {}

for model_name, hf in models.items():   
    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(hf)
    model = AutoModelForCausalLM.from_pretrained(
        hf,
        dtype="auto",
        device_map=None
    ).to(device)

    for prompt in ["structured", "semi-structured", "unstructured"]:
        for row in tqdm(triples.iterrows(), total=len(triples)):
            print(run_pipeline(model, tokenizer, r_isco, r_esco, row[1], model_name, prompt))
        
        del model
        del tokenizer
        torch.cuda.empty_cache() 
        torch.cuda.ipc_collect()
        gc.collect()

triples.head()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  0%|          | 0/10584 [00:00<?, ?it/s]

{'isco': '[("IT-administrator", "has_isco", "2522")]', 'esco': '[("IT-administrator", "has_esco", "011151"), ("IT-administrator", "has_esco", "004542"), ("IT-administrator", "has_esco", "011116")]'}
{'isco': '[("IT-administrator", "has_isco", "2522")]', 'esco': '[("business ICT systems", "has_esco", "011151")]'}
{'isco': '[("SQE Manager", "has_isco", "1420")]', 'esco': '[("Risk Assessment", "has_esco", "000024"), ("Problem Solving", "has_esco", "011009"), ("Attention to Detail", "has_esco", "008163")]'}
{'isco': '[("Kundeservice / teknisk support", "has_isco", "3512")]', 'esco': '[("Kundeservice / teknisk support", "has_esco", "010600")]'}
{'isco': '[]', 'esco': '[]'}
{'isco': '[]', 'esco': '[("problem-solving", "has_esco", "010923")]'}
{'isco': '[]', 'esco': '[]'}
{'isco': '[]', 'esco': '[]'}
{'isco': '[]', 'esco': '[]'}
{'isco': '[("Lægesekretær/blæksprutte", "has_isco", "4120")]', 'esco': '[]'}
{'isco': '[("Shippingmedarbejder", "has_isco", "6121)]', 'esco': '[("Shippingmedarbejder"

KeyboardInterrupt: 

In [ ]:
triples_ISCO_ESCO[["triples", "ISCO", "ESCO"]].head()